# OpenEvolve TSP Demo —— 從頭跑到尾(教學版)

由上往下逐格按 ▶ 執行即可。

> **環境**:Windows + Python 3.10 以上。Jupyter 或 VS Code 開都可以。


## 第 0 步:安裝 Ollama(本地 LLM 執行器)

用 Windows 內建的 winget 自動安裝。可能跳出安裝視窗,照預設裝完即可。
> 若報「找不到 winget」,請改到 https://ollama.com 下載 Windows 版手動安裝。
> **裝完不用重啟 kernel**,直接跑下一格(第 0.5 步)會自動找到它。


In [ ]:
!winget install Ollama.Ollama --accept-source-agreements --accept-package-agreements

## 第 0.5 步:讓系統找到 Ollama(自動,不用重啟 kernel)

Ollama 裝好後,有時 kernel 還不認得它的指令。
這格會**自動掃描常見安裝位置、找到就加進 PATH**,跑完就能用 ollama 指令。
- 印 ✅ → 繼續下一步
- 印 ❌ → 請確認上一步真的裝好了,或到 https://ollama.com 手動安裝


In [ ]:
import os, glob, shutil

# 先看現在認不認得
if shutil.which("ollama"):
    print("✅ Ollama 已就緒:", shutil.which("ollama"))
else:
    # 自動掃描常見安裝位置
    patterns = [
        os.path.expandvars(r"%LOCALAPPDATA%\\Programs\\Ollama\\ollama.exe"),
        r"C:\\Program Files\\Ollama\\ollama.exe",
        r"C:\\Program Files (x86)\\Ollama\\ollama.exe",
    ]
    found = [p for p in patterns if os.path.exists(p)]
    if not found:
        # 更廣的搜尋(使用者目錄底下)
        found = glob.glob(os.path.expandvars(r"%USERPROFILE%\\**\\ollama.exe"), recursive=True)

    if found:
        ollama_dir = os.path.dirname(found[0])
        os.environ["PATH"] += os.pathsep + ollama_dir
        print("✅ 已自動加入 PATH:", found[0])
    else:
        print("❌ 找不到 Ollama。請確認第 0 步真的安裝成功,")
        print("   或到 https://ollama.com 手動安裝 Windows 版後再跑這格。")

## 第 1 步:下載模型(只需做一次,約 4.7GB)

下載 qwen2.5-coder:7b(千問程式模型)。第一次會比較久,請耐心等到出現 `success`。

> 這個模型在「小模型寫 diff」這件事上最穩,演化才跑得動。
> 顯卡夠力(20GB+)可改成 `qwen2.5-coder:14b`(config.yaml 也要一起改)。


In [ ]:
!ollama pull qwen2.5-coder:7b

## 第 2 步:安裝 Python 套件

用 kernel 自己的 Python 安裝(避免裝錯環境),裝完會**列出實際安裝的套件與版本**。

主要會裝:
- `openevolve` — 演化框架本體
- `matplotlib` — 畫 TSP 路線圖
- (連帶裝它們的相依套件,例如 `openai`、`numpy`、`pyyaml` 等)


In [ ]:
import sys, subprocess

# 用「當前 kernel 的同一個 Python」安裝,避免裝錯環境
print("使用的 Python:", sys.executable)
print("=" * 50)
print("開始安裝套件(會即時顯示進度)...")
print("=" * 50)

# 重點:encoding="utf-8" + errors="replace"
# 強制用 UTF-8 讀取輸出,避開 Windows cp950(Big5)解碼錯誤
proc = subprocess.Popen(
    [sys.executable, "-m", "pip", "install", "-r", "requirements.txt"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, encoding="utf-8", errors="replace", bufsize=1
)
for line in proc.stdout:
    print(line, end="")
proc.wait()

print("\n" + "=" * 50)
print("=== 安裝結果:關鍵套件與版本 ===")
for pkg in ["openevolve", "matplotlib", "numpy", "openai", "pyyaml"]:
    r = subprocess.run([sys.executable, "-m", "pip", "show", pkg],
                       capture_output=True, text=True,
                       encoding="utf-8", errors="replace")
    if r.returncode == 0:
        name = ver = ""
        for line in r.stdout.splitlines():
            if line.startswith("Name:"):
                name = line.split(":", 1)[1].strip()
            elif line.startswith("Version:"):
                ver = line.split(":", 1)[1].strip()
        print(f"  OK  {name:14} {ver}")
    else:
        print(f"  XX  {pkg:14} 未安裝")

print("\n套件安裝完成")

## 第 2.5 步:確認連得到 Ollama 服務

OpenEvolve 透過 localhost:11434 連本機 Ollama。
成功印 ✅;失敗就開終端機執行 `ollama serve` 放著別關,再重跑這格。


In [ ]:
import urllib.request
try:
    with urllib.request.urlopen("http://localhost:11434/api/tags", timeout=5) as r:
        print("✅ Ollama 服務正常,連得到 localhost:11434")
except Exception as e:
    print("❌ 連不到 Ollama。請開終端機執行  ollama serve  再重跑這格。")
    print("   錯誤:", e)

## 第 3 步:先測評分(不燒 LLM,確認檔案沒問題)

**預期**:分數約 `0.75`,看到一條亂繞、有交叉的初始路線。


In [ ]:
import os
import matplotlib.pyplot as plt
from evaluator import CITIES, _tour_length, evaluate
from initial_program import solve

result = evaluate(os.path.join(os.getcwd(), "initial_program.py"))
print("評分結果:", result)

tour = solve(CITIES)
length = _tour_length(tour)

def plot_tour(tour, title):
    xs = [CITIES[i][0] for i in tour] + [CITIES[tour[0]][0]]
    ys = [CITIES[i][1] for i in tour] + [CITIES[tour[0]][1]]
    plt.figure(figsize=(6, 6))
    plt.plot(xs, ys, "-o", linewidth=1.5, markersize=6)
    plt.title(title)
    plt.show()

plot_tour(tour, f"初始路線(最近鄰法)— 總長 {length:.1f}")

## 第 4 步:正式啟動演化

讓本地 LLM 一代代改寫求解程式。每代印出分數(`combined_score`,越高越好)。
> ⏳ 視顯卡而定需數分鐘。中途 `route.png` 會即時更新。


In [ ]:
import os, sys

# notebook 已有 event loop,需要 nest_asyncio 才能跑 openevolve 的 asyncio
try:
    import nest_asyncio
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "nest_asyncio", "-q"],
                   encoding="utf-8", errors="replace")
    import nest_asyncio
nest_asyncio.apply()

from openevolve import run_evolution

HERE = os.getcwd()
OUT = os.path.join(HERE, "openevolve_output")   # 結果存在專案資料夾(不要丟暫存)

result = run_evolution(
    initial_program=os.path.join(HERE, "initial_program.py"),
    evaluator=os.path.join(HERE, "evaluator.py"),
    config=os.path.join(HERE, "config.yaml"),
    output_dir=OUT,
    cleanup=False,        # 保留輸出,第 5 步才讀得到
)
print("\n✅ 演化完成!最佳程式存在:", OUT)

## 第 5 步:看演化後的最佳路線,跟初始比較

**預期**:總長度變短,路線變成乾淨漂亮的環、交叉消失。


In [ ]:
import os, importlib.util
import matplotlib.pyplot as plt
from evaluator import CITIES, _tour_length
from initial_program import solve as _init_solve

HERE = os.getcwd()

# 初始長度(對比用)
init_tour = _init_solve(CITIES)
init_length = _tour_length(init_tour)

# 載入演化後的最佳程式(從專案資料夾的 openevolve_output)
best_path = os.path.join(HERE, "openevolve_output", "best", "best_program.py")
if not os.path.exists(best_path):
    print("找不到最佳程式,請先成功跑完第 4 步。路徑:", best_path)
else:
    spec = importlib.util.spec_from_file_location("best", best_path)
    best = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(best)

    best_tour = best.solve(CITIES)
    best_length = _tour_length(best_tour)

    print(f"初始路線長度:{init_length:.1f}")
    print(f"演化後長度:  {best_length:.1f}")
    diff = init_length - best_length
    print(f"縮短了:      {diff:.1f}  ({diff/init_length*100:.1f}%)")
    if diff < 1:
        print("\n(註:若幾乎沒縮短,可能是模型沒寫出有效改進,可多跑幾代或加大 max_iterations)")

    def plot_tour(tour, title):
        xs = [CITIES[i][0] for i in tour] + [CITIES[tour[0]][0]]
        ys = [CITIES[i][1] for i in tour] + [CITIES[tour[0]][1]]
        plt.figure(figsize=(6, 6))
        plt.plot(xs, ys, "-o", linewidth=1.5, markersize=6)
        plt.title(title)
        plt.show()

    plot_tour(best_tour, f"演化後最佳路線 — 總長 {best_length:.1f}")

## 想再深入?

- 打開 `openevolve_output/best/best_program.py`,看 LLM 加了什麼技巧(2-opt、模擬退火等)。
- 改 `config.yaml` 的 `max_iterations` 跑更多代。

---
### 常見問題
| 症狀 | 解法 |
|---|---|
| 第 0.5 步印 ❌ | 確認 Ollama 真的裝好;或到 ollama.com 手動裝 |
| 第 2.5 步印 ❌ | 開終端機執行 `ollama serve` |
| 第 4 步卡住沒分數 | 同上,Ollama 服務沒開 |
| `No valid diffs found` | 模型偶爾不照格式,多跑幾代通常恢復 |
